# 03 Gold Layer: Business Aggregations & Analytical KPIs

The **Gold Layer** provides curated, query-optimized analytical models ready for:
- BI Dashboards & Executive KPI reporting
- Demand forecasting & fleet optimization
- Transit corridor and congestion pricing analysis

### Output Gold Tables:
1. `workspace.taxi_gold.daily_metrics`: Daily volume, revenue, tips, and payment method distribution by borough.
2. `workspace.taxi_gold.zone_performance`: High-density pickup -> dropoff transit corridors and route economics.
3. `workspace.taxi_gold.hourly_demand`: Rush-hour patterns, average travel speeds, and weekday vs weekend demand.


In [0]:
import sys
# Add src to Python path for Databricks workspace execution or local session
sys.path.append("/Workspace/Users/yahyaghallali88@gmail.com/nyc_taxi-DB-DE/src")
sys.path.append("./src")

from pyspark.sql import functions as F
from taxi_lakehouse.gold import build_daily_metrics, build_zone_performance, build_hourly_demand


## 1. Load Silver Cleaned Data


In [0]:
silver_table = "workspace.taxi_silver.yellow_trips"
df_silver = spark.read.table(silver_table)

print(f"Loaded {df_silver.count():,} cleaned records from {silver_table}")
display(df_silver.limit(5))


## 2. Gold Table 1: Daily Operational & Financial Metrics (`daily_metrics`)
Aggregates daily trip volume, total revenue, average fare, average tip %, and payment splits by pickup borough.


In [0]:
df_daily = build_daily_metrics(df_silver)
daily_table = "workspace.taxi_gold.daily_metrics"

(
    df_daily.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(daily_table)
)

print(f"Saved daily metrics to {daily_table}")
display(df_daily.limit(20))


## 3. Gold Table 2: Zone Corridor & Route Performance (`zone_performance`)
Aggregates route volume, total revenue, average duration, distance, and fares between pickup and dropoff zones.


In [0]:
# Routes with at least 10 trips
df_zone_perf = build_zone_performance(df_silver, min_trips=10)
zone_perf_table = "workspace.taxi_gold.zone_performance"

(
    df_zone_perf.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(zone_perf_table)
)

print(f"Saved zone performance to {zone_perf_table}")
display(df_zone_perf.limit(20))


## 4. Gold Table 3: Hourly Demand & Traffic Dynamics (`hourly_demand`)
Aggregates hourly trip volume, average speeds (mph), duration, and fare amounts across weekdays vs weekends.


In [0]:
df_hourly = build_hourly_demand(df_silver)
hourly_table = "workspace.taxi_gold.hourly_demand"

(
    df_hourly.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(hourly_table)
)

print(f"Saved hourly demand to {hourly_table}")
display(df_hourly.orderBy("is_weekend", "pickup_hour"))


## 5. Executive KPI Summary Dashboard
Consolidated high-level executive metrics computed from Gold tables.


In [0]:
display(spark.sql(f"""
    SELECT 
        'Total Revenue' AS KPI,
        CONCAT('$', FORMAT_NUMBER(SUM(total_revenue), 2)) AS Metric_Value
    FROM {daily_table}
    UNION ALL
    SELECT 
        'Total Trips',
        FORMAT_NUMBER(SUM(total_trips), 0)
    FROM {daily_table}
    UNION ALL
    SELECT 
        'Total Passengers Carried',
        FORMAT_NUMBER(SUM(total_passengers), 0)
    FROM {daily_table}
    UNION ALL
    SELECT 
        'Average Fare Amount',
        CONCAT('$', ROUND(AVG(avg_fare_amount), 2))
    FROM {daily_table}
    UNION ALL
    SELECT 
        'Average Tip % (Credit Card Trips)',
        CONCAT(ROUND(AVG(avg_tip_pct), 2), '%')
    FROM {daily_table}
"""))
